In [27]:
import os
import glob
import pandas as pd
from pathlib import Path

# 1. Find where this notebook is saved
try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path(os.getcwd()).resolve()

# 2. Locate FinalProject whether the kernel starts in notebooks, FinalProject,
# or the top-level cosmos workspace.
project_candidates = [
    current_dir.parent if current_dir.name == 'notebooks' else current_dir,
    current_dir / '26-the-deep-learners-analysis' / 'FinalProject',
]
project_root = next(
    (path for path in project_candidates if (path / 'data' / 'raw').exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate FinalProject/data/raw from the current directory.')

input_folder = project_root / "data" / "interim"
output_folder = project_root / "data" / "processed"
output_filename = "merged_features.csv"

# Create the processed folder automatically if it's missing
os.makedirs(output_folder, exist_ok=True)

# 3. Grab all CSV files inside that precise folder
search_path = os.path.join(input_folder, "*.csv")
csv_files = glob.glob(search_path)
csv_files = [f for f in csv_files if os.path.basename(f) != output_filename]

print(f"Project root directory: {project_root}")
print(f"Looking inside input path: {input_folder}")
print(f"Successfully found {len(csv_files)} file(s) to merge!")


Project root directory: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject
Looking inside input path: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim
Successfully found 10 file(s) to merge!


In [28]:
if len(csv_files) < 2:
    print(f"Found {len(csv_files)} file(s) in '{input_folder}'. You need at least 2 files to merge.")
    print("Ensure your VS Code terminal is open to the root folder containing 'data'.")
    exit()

# 2. Open the first file to establish the baseline
print(f"Reading base file: {csv_files[0]}")
master_df = pd.read_csv(csv_files[0])

# Automatically detect the names of your first two tracking columns
id_names = list(master_df.columns[:2])
print(f"Tracking keys detected: {id_names}")



Reading base file: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\additional_pair_features.csv
Tracking keys detected: ['user_a', 'user_b']


In [29]:
# 3. Loop through and merge the remaining files sideways
for file in csv_files[1:]:
    print(f"Merging: {file}")
    df_next = pd.read_csv(file)
    
    # Clean up column names to prevent case or space mismatches
    df_next.columns = df_next.columns.str.strip().str.lower()
    
    # Double check if both id_names are present in this specific file
    missing_keys = [key for key in id_names if key not in df_next.columns]
    if missing_keys:
        print(f"⚠️ Warning: Skipping {file} because it is missing tracking key columns: {missing_keys}")
        print(f"   Available columns in this file: {list(df_next.columns)}")
        continue  # Safely skip this broken file and move to the next one
        
    master_df = pd.merge(master_df, df_next, on=id_names, how='outer')

# Requested pair-level features are calculated below before the final CSV is saved.


Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_days.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_totals.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pairwise_features.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_facebook_friend_counts.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_call_streaks.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_text_streaks.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\interim\pair_total_texts_sent.csv
Merging: C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\d

In [30]:
import numpy as np
from itertools import combinations

SECONDS_PER_DAY = 24 * 60 * 60
pair_keys = ['user_a', 'user_b']
raw_folder = project_root / 'data' / 'raw'

# Read the raw interaction logs. Some CNS files prefix the first header with '#'.
calls = pd.read_csv(raw_folder / 'calls.csv')
sms = pd.read_csv(raw_folder / 'sms.csv')
bluetooth = pd.read_csv(raw_folder / 'bt_symmetric.csv')
for frame in (calls, sms, bluetooth):
    frame.columns = frame.columns.str.strip().str.lstrip('#').str.strip().str.lower()

# Canonicalize the already-merged keys so every unordered pair is user_a < user_b.
master_df[pair_keys] = master_df[pair_keys].apply(pd.to_numeric, errors='coerce')
master_df = master_df.dropna(subset=pair_keys)
master_df[['user_a', 'user_b']] = np.sort(master_df[pair_keys].astype('int64'), axis=1)
master_df = master_df.loc[master_df['user_a'].ne(master_df['user_b'])]

# If an interim table contains repeated pair rows, retain the first non-null value
# in each feature column instead of multiplying rows during later joins.
master_df = master_df.groupby(pair_keys, as_index=False, sort=False).first()

# Build the complete pair universe from every valid participant seen in the
# merged features, call logs, or SMS logs. Bluetooth user_b values can include
# non-participant device IDs, so Bluetooth is intentionally not a user roster.
participant_series = [
    master_df['user_a'], master_df['user_b'],
    calls['caller'], calls['callee'],
    sms['sender'], sms['recipient'],
]
all_users = (
    pd.to_numeric(pd.concat(participant_series, ignore_index=True), errors='coerce')
    .dropna().astype('int64')
)
all_users = sorted(all_users.loc[all_users.ge(0)].unique())
all_pairs = pd.DataFrame(combinations(all_users, 2), columns=pair_keys)
master_df = all_pairs.merge(master_df, on=pair_keys, how='left', validate='one_to_one')

def canonical_interactions(frame, left_user, right_user):
    """Return timestamped, valid, unordered participant-pair interactions."""
    events = frame[['timestamp', left_user, right_user]].rename(
        columns={left_user: 'left_user', right_user: 'right_user'}
    ).copy()
    for column in events.columns:
        events[column] = pd.to_numeric(events[column], errors='coerce')
    events = events.dropna().astype('int64')
    events = events.loc[
        events['left_user'].ge(0)
        & events['right_user'].ge(0)
        & events['left_user'].ne(events['right_user'])
        & events['left_user'].isin(all_users)
        & events['right_user'].isin(all_users)
    ]
    events['user_a'] = events[['left_user', 'right_user']].min(axis=1)
    events['user_b'] = events[['left_user', 'right_user']].max(axis=1)
    return events[['timestamp', 'user_a', 'user_b']]

# Weekend-vs-weekday ratio for interactions between the two users. Use distinct
# active pair-days so frequent Bluetooth scans do not swamp calls and texts.
interaction_events = pd.concat(
    [
        canonical_interactions(calls, 'caller', 'callee'),
        canonical_interactions(sms, 'sender', 'recipient'),
        canonical_interactions(bluetooth, 'user_a', 'user_b'),
    ],
    ignore_index=True,
)
interaction_events['elapsed_day'] = interaction_events['timestamp'] // SECONDS_PER_DAY
active_pair_days = interaction_events[pair_keys + ['elapsed_day']].drop_duplicates()
active_pair_days['is_weekend'] = active_pair_days['elapsed_day'].mod(7).isin([0, 6])

week_part = (
    active_pair_days.groupby(pair_keys)['is_weekend']
    .agg(weekend_interaction_days='sum', total_interaction_days='size')
    .reset_index()
)
week_part['weekday_interaction_days'] = (
    week_part['total_interaction_days'] - week_part['weekend_interaction_days']
)
week_part['weekend_weekday_interaction_ratio'] = (
    week_part['weekend_interaction_days']
    .div(week_part['weekday_interaction_days'].replace(0, np.nan))
)
master_df = master_df.drop(columns=['weekend_weekday_interaction_ratio'], errors='ignore')
master_df = master_df.merge(
    week_part[pair_keys + ['weekend_weekday_interaction_ratio']],
    on=pair_keys,
    how='left',
    validate='one_to_one',
)

# Per-day call totals include both members' behavior with anyone. A missed call
# is an incoming call with duration -1.
max_timestamp = max(calls['timestamp'].max(), sms['timestamp'].max(), bluetooth['timestamp'].max())
total_observation_days = int(max_timestamp // SECONDS_PER_DAY) + 1

# Recompute combined daily texts received for every pair in the expanded
# universe. Values inherited from additional_pair_features only cover its
# smaller candidate-pair table, which leaves the newly created pairs blank.
combined_text_column = 'combined_daily_texts_received'
missing_combined_before = (
    int(master_df[combined_text_column].isna().sum())
    if combined_text_column in master_df.columns else len(master_df)
)
texts_received_per_day = (
    sms.groupby('recipient').size() / total_observation_days
)
master_df[combined_text_column] = (
    master_df['user_a'].map(texts_received_per_day).fillna(0.0)
    + master_df['user_b'].map(texts_received_per_day).fillna(0.0)
)
assert master_df[combined_text_column].notna().all()
print(
    f'Corrected {missing_combined_before:,} missing '
    f'{combined_text_column} values.'
)
calls_received = calls.groupby('callee').size()
calls_sent = calls.groupby('caller').size()
missed_received = calls.loc[calls['duration'].eq(-1)].groupby('callee').size()

per_user_rates = {
    'calls_received_per_day': calls_received / total_observation_days,
    'calls_sent_per_day': calls_sent / total_observation_days,
    'missed_calls_per_day': missed_received / total_observation_days,
}

# Remove any older per-member versions supplied by an interim feature file.
individual_rate_columns = [
    f'{feature_name}_{member}'
    for feature_name in per_user_rates
    for member in ('a', 'b')
]
master_df = master_df.drop(columns=individual_rate_columns, errors='ignore')

for feature_name, rates in per_user_rates.items():
    user_a_rate = master_df['user_a'].map(rates).fillna(0.0)
    user_b_rate = master_df['user_b'].map(rates).fillna(0.0)
    master_df[f'total_{feature_name}'] = user_a_rate + user_b_rate

print(f'Created requested features for {len(master_df):,} pairs across {total_observation_days} days.')


Corrected 268,690 missing combined_daily_texts_received values.
Created requested features for 351,541 pairs across 28 days.


In [31]:
pd.set_option('display.max_columns', None)
print(master_df.head())

   user_a  user_b  in_person_contact_minutes  user_a_daily_texts_received  \
0       0       1                        NaN                          NaN   
1       0       2                        NaN                          NaN   
2       0       3                        0.0                     2.642857   
3       0       4                        NaN                          NaN   
4       0       5                        0.0                     2.642857   

   user_b_daily_texts_received  combined_daily_texts_received  \
0                          NaN                       2.678571   
1                          NaN                       2.642857   
2                     5.250000                       7.892857   
3                          NaN                       5.107143   
4                     0.642857                       3.285714   

   minimum_call_contact_time_seconds has_completed_call is_fb_friend  \
0                                NaN               None         None   
1 

In [32]:
requested_columns = [
    'combined_daily_texts_received',
    'weekend_weekday_interaction_ratio',
    'total_calls_received_per_day',
    'total_calls_sent_per_day',
    'total_missed_calls_per_day',
]
assert not master_df[pair_keys].duplicated().any()
assert len(master_df) == len(all_users) * (len(all_users) - 1) // 2
assert set(requested_columns).issubset(master_df.columns)
assert master_df['combined_daily_texts_received'].notna().all()

# Save only after the requested columns have been calculated and validated.
destination_path = output_folder / output_filename
master_df.to_csv(destination_path, index=False)

print(f"Success! Saved {len(master_df):,} pair rows to '{destination_path}'")
display(master_df[pair_keys + requested_columns].head())


Success! Saved 351,541 pair rows to 'C:\Users\Rmcas\OneDrive\Desktop\COSMOS\26-the-deep-learners-analysis\FinalProject\data\processed\merged_features.csv'


,user_a,user_b,combined_daily_texts_received,weekend_weekday_interaction_ratio,total_calls_received_per_day,total_calls_sent_per_day,total_missed_calls_per_day
0,0,1,2.678571,NaN,0.214286,0.107143,0.000000
1,0,2,2.642857,NaN,0.214286,0.107143,0.000000
2,0,3,7.892857,0.0,0.571429,0.500000,0.035714
3,0,4,5.107143,NaN,0.928571,0.964286,0.000000
4,0,5,3.285714,0.0,0.392857,0.142857,0.107143


In [33]:
columns_to_remove = ['user_a_daily_texts_received', 'user_b_daily_texts_received', 'has_completed_call', 'is_fb_friend', 'fraction_rssi_above_threshold', 'fraction_rssi_below_threshold', 'active_days', 'longest_consecutive_days', 'mutual_friends', 'proximity_measurements', 'weekend_interaction_fraction', 'weekday_interaction_fraction']
master_df = master_df.drop(columns=columns_to_remove, errors='ignore')

In [34]:
# Print each column with its index number
for index, col in enumerate(master_df.columns, start=1):
    print(f"{index}. {col}")

print(f"\nTotal number of columns: {len(master_df.columns)}")


1. user_a
2. user_b
3. in_person_contact_minutes
4. combined_daily_texts_received
5. minimum_call_contact_time_seconds
6. days_called
7. total_calls
8. average_proximity_rssi
9. texts_shared
10. calls_shared
11. average_call_contact_time
12. max_call_contact_time
13. text_reciprocity
14. call_reciprocity
15. total_facebook_friend_count
16. longest_consecutive_call_streak_days
17. longest_consecutive_text_streak_days
18. total_texts_sent
19. days_texted
20. weekend_weekday_interaction_ratio
21. total_calls_received_per_day
22. total_calls_sent_per_day
23. total_missed_calls_per_day

Total number of columns: 23


In [35]:
print(master_df.head())

   user_a  user_b  in_person_contact_minutes  combined_daily_texts_received  \
0       0       1                        NaN                       2.678571   
1       0       2                        NaN                       2.642857   
2       0       3                        0.0                       7.892857   
3       0       4                        NaN                       5.107143   
4       0       5                        0.0                       3.285714   

   minimum_call_contact_time_seconds  days_called  total_calls  \
0                                NaN          NaN          NaN   
1                                NaN          NaN          NaN   
2                                NaN          NaN          NaN   
3                                NaN          NaN          NaN   
4                                NaN          NaN          NaN   

   average_proximity_rssi  texts_shared  calls_shared  \
0                     NaN           NaN           NaN   
1             